### Middleware 
Middleware provides a way to more tightly control what happens inside the agents. Middleware is useful for the following:
- Tracking agent behavior with logging , analytics and debugging.
- Transforming prompts , tool selection and output formatting.
- Adding retries , fallbacks and ealry termination logic.
- Applying rate limits , guradrails and PII detection.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["Gemini_API_Key"] = os.getenv("Gemini_API_Key")

## Summarization Middleware 
- Automatically summarizes conversation history when approching token limits, preserving recent messge while compressing older context . Summarization is usefull for the following:
- Long-runing conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage , AIMessage , SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["Gemini_API_Key"] = os.getenv("Gemini_API_Key")



model = ChatGoogleGenerativeAI(
      model="gemini-2.5-flash",
      google_api_key=os.getenv("Gemini_API_Key")
)

agent = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [26]:
## Run with thread id
config = {"configurable":{"thread_id":"test-1"}}

In [27]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?"
    "what is 15-7?",
    "what is 4*4?"
    "What is 16*8?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages :{response}")
    print(f"Message:{len(response['messages'])}")

Messages :{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3942dd26-1c2d-420e-821e-c5ab6f182f2f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef842-b994-7583-868a-ca55724647c6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 18}})]}
Message:2
Messages :{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3942dd26-1c2d-420e-821e-c5ab6f182f2f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef842-b994-7583-868a-ca55724647c6-0', tool_calls=[

In [ ]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage , AIMessage , SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

@tool
def search_hotels(city:str) ->str:
    """Search Hotels - returns long response to use more token"""
    return f"""Hotels in {city}
1. Grand Hotal - 5 star , $350/night , spa , pool , gym
2. City Inn - 4 star , $180/night , business center 
3. Budget Stay - 3 star , $75/night , free wifi 
"""

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",10),
            keep = ("tokens",10)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

In [5]:
cities = ["Paris","London","New York","Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens , {len(response["messages"])} messages")
    print(f"{(response["messages"])}")

Paris: ~993 tokens , 4 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to find hotels in Paris.\n\n## SUMMARY\n\nThe user initiated the conversation with a request to find hotels in Paris. No further details, choices, or strategies have been discussed yet.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nThe next step is to begin searching for hotels in Paris. This may involve asking the user for additional criteria (e.g., dates, price range, specific amenities, number of guests) to refine the search.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='cde1f1a0-0cd1-4bbf-ad79-e775bc2fb0e5'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'5a956323-46fb-4d47-a77c-954bd2d772f0': 'CrICAQw51sfs86VAolkVM2FfsCoNVOzHltO0XiBKK7qZ8yx5SKJfu87sacbDBP4KeaA7Q0zaCyz1Cq3VNPZZfsWMKO